
# RT Notebook 17 — Organization → Resolution → Distinction Density

**Campaign ID:** `MPF_SIM_ORGANIZATION_RESOLUTION_CALCULUS_001`

## Central distinction

This notebook does **not** identify organization with distinction density.

\[
\mathcal O
\;\xrightarrow{\;\text{resolution relative to }\langle S\rangle\;}
\delta D_{\langle S\rangle}
\]

- \(\mathcal O\): reference-independent organization candidate.
- \(\langle S\rangle\): an operational symmetry/reference condition.
- \(\delta D_{\langle S\rangle}\): distinction density resolved under that reference.

The same organization may resolve to different distinction-density measurements under different references.

## Primary questions

1. Can a stable organization quantity be extracted from Notebook 16 without reducing it to graph size?
2. Does that quantity retain predictive value after controlling for ordinary graph invariants?
3. Does projection generally preserve or reduce organization?
4. Can reference changes alter measured distinction density while leaving organization unchanged?
5. Which observations falsify the proposed organization axioms?

## Required input

Upload:

`MPF_SIM_PROJECTION_DOF_MEANINGFUL_001_RESULTS.zip`

No other file is required.

## Evidence boundary

All conclusions are bounded to:

- the Notebook 16 expression-graph encoding;
- its 527 catalog organizations;
- the supplied perturbation and projection tables;
- the operational reference conditions defined here.

This notebook does not establish a formal organization calculus or external physical validity.


In [ ]:

#@title 1. Configuration
from pathlib import Path

CAMPAIGN_ID = "MPF_SIM_ORGANIZATION_RESOLUTION_CALCULUS_001"
SEED = 170017
CV_SPLITS = 5
BOOTSTRAP_REPEATS = 1000
OUTPUT_DIR = Path("/content/rt_notebook_17_results")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

INPUT_CANDIDATES = [
    Path("/content/MPF_SIM_PROJECTION_DOF_MEANINGFUL_001_RESULTS.zip"),
    Path("/mnt/data/MPF_SIM_PROJECTION_DOF_MEANINGFUL_001_RESULTS.zip"),
]

print("Campaign:", CAMPAIGN_ID)
print("Output:", OUTPUT_DIR)



## Upload instructions

1. Open this notebook in Google Colab.
2. Run the cells in order.
3. At the upload prompt, choose  
   `MPF_SIM_PROJECTION_DOF_MEANINGFUL_001_RESULTS.zip`.
4. Do **not** unzip it first.
5. At the final cell, Colab downloads  
   `MPF_SIM_ORGANIZATION_RESOLUTION_CALCULUS_001_RESULTS.zip`.

If the ZIP is already in `/content`, the upload prompt is skipped.


In [ ]:

#@title 2. Imports and upload
import hashlib
import json
import math
import random
import re
import shutil
import zipfile
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import spearmanr, mannwhitneyu
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import (
    balanced_accuracy_score, r2_score, roc_auc_score
)
from sklearn.model_selection import (
    GroupKFold, KFold, StratifiedKFold, cross_val_predict
)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

rng = np.random.default_rng(SEED)
random.seed(SEED)

source_zip = next((p for p in INPUT_CANDIDATES if p.exists()), None)

if source_zip is None:
    try:
        from google.colab import files
        print("Upload MPF_SIM_PROJECTION_DOF_MEANINGFUL_001_RESULTS.zip")
        uploaded = files.upload()
        if not uploaded:
            raise FileNotFoundError("No file uploaded.")
        name, payload = next(iter(uploaded.items()))
        source_zip = Path("/content") / name
        source_zip.write_bytes(payload)
    except ImportError as exc:
        raise FileNotFoundError(
            "Input ZIP not found. Place it in /content or /mnt/data."
        ) from exc

EXTRACT_DIR = OUTPUT_DIR / "input"
if EXTRACT_DIR.exists():
    shutil.rmtree(EXTRACT_DIR)
EXTRACT_DIR.mkdir(parents=True)

with zipfile.ZipFile(source_zip) as zf:
    zf.extractall(EXTRACT_DIR)

print("Loaded:", source_zip)


In [ ]:

#@title 3. Validate the Notebook 16 result bundle

REQUIRED_FILES = {
    "manifest.json",
    "organization_features.csv",
    "normalized_effects.csv",
    "perturbation_trials.csv",
    "projection_loss_analyzed.csv",
    "robustness_summary.csv",
    "decision_record.json",
}

present = {p.name for p in EXTRACT_DIR.iterdir() if p.is_file()}
missing = REQUIRED_FILES - present
if missing:
    raise ValueError(f"Input archive is missing: {sorted(missing)}")

source_manifest = json.loads((EXTRACT_DIR / "manifest.json").read_text())
if source_manifest.get("status") != "EXECUTED":
    raise ValueError(
        "Notebook 16 manifest is not marked EXECUTED. "
        "Use the actual result archive, not the source specification."
    )

features = pd.read_csv(EXTRACT_DIR / "organization_features.csv")
effects = pd.read_csv(EXTRACT_DIR / "normalized_effects.csv")
perturbations = pd.read_csv(EXTRACT_DIR / "perturbation_trials.csv")
projection = pd.read_csv(EXTRACT_DIR / "projection_loss_analyzed.csv")

print(json.dumps({
    "source_campaign": source_manifest.get("campaign_id"),
    "status": source_manifest.get("status"),
    "organizations": len(features),
    "effect_rows": len(effects),
    "perturbation_rows": len(perturbations),
    "projection_rows": len(projection),
}, indent=2))



## 1. Constructing organization candidates

Notebook 16 measured deviations from size-matched random controls.  
These standardized deviations are useful because they do not merely count nodes or edges.

Let \(z_i(A)\) be the matched-null standardized deviation of organization \(A\) on structural feature \(i\).

This notebook tests several organization candidates:

\[
\mathcal O_{L1}(A)=\frac{1}{n}\sum_i |z_i(A)|
\]

\[
\mathcal O_{L2}(A)=\sqrt{\frac{1}{n}\sum_i z_i(A)^2}
\]

\[
\mathcal O_{\max}(A)=\max_i |z_i(A)|
\]

and a latent organization coordinate obtained from PCA.

None is assumed to be the true organization quantity. They are competing operational candidates.


In [ ]:

#@title 4. Build the organization feature space

z_cols = [c for c in effects.columns if c.endswith("__z")]
if not z_cols:
    raise ValueError("No matched-null z-score columns found.")

z_names = [c.removesuffix("__z") for c in z_cols]
Z = effects[z_cols].replace([np.inf, -np.inf], np.nan)
Z = pd.DataFrame(
    SimpleImputer(strategy="constant", fill_value=0.0).fit_transform(Z),
    columns=z_names,
    index=effects.index,
)

# Standardized shape coordinates prevent one metric scale from dominating PCA.
Z_scaled = StandardScaler().fit_transform(Z)

pca = PCA(random_state=SEED)
scores = pca.fit_transform(Z_scaled)

organization = effects[
    ["organization_id", "dof", "signature",
     "novel_to_lower_dof", "closed", "stable"]
].copy()

organization["O_L1"] = Z.abs().mean(axis=1)
organization["O_L2"] = np.sqrt((Z ** 2).mean(axis=1))
organization["O_MAX"] = Z.abs().max(axis=1)
organization["O_PCA1_raw"] = scores[:, 0]

# Orient PCA1 so that it is positively associated with O_L1.
if spearmanr(organization["O_PCA1_raw"], organization["O_L1"]).statistic < 0:
    organization["O_PCA1"] = -organization["O_PCA1_raw"]
else:
    organization["O_PCA1"] = organization["O_PCA1_raw"]

explained = pd.DataFrame({
    "component": np.arange(1, len(pca.explained_variance_ratio_) + 1),
    "explained_variance_ratio": pca.explained_variance_ratio_,
    "cumulative": np.cumsum(pca.explained_variance_ratio_),
})

print("Matched-null structural dimensions:", len(z_names))
display(explained.head(10))
display(organization.head())


In [ ]:

#@title 5. Attach robustness and ordinary graph invariants

robustness = perturbations.groupby(["organization_id", "dof"]).agg(
    perturbation_retention=("feature_retention", "mean"),
    perturbation_retention_sd=("feature_retention", "std"),
    connected_after_rate=("connected_after", "mean"),
).reset_index()

GRAPH_CONTROLS = [
    "node_count_graph",
    "edge_count_graph",
    "density_graph",
    "max_depth_graph",
    "degree_entropy",
    "label_entropy",
    "articulation_fraction",
    "bridge_fraction",
    "scc_count",
    "transitivity",
]

missing_controls = [c for c in GRAPH_CONTROLS if c not in features.columns]
if missing_controls:
    raise ValueError(f"Missing graph controls: {missing_controls}")

data = (
    organization
    .merge(features[
        ["organization_id", "dof", "symmetry_class",
         "primitive_preserved"] + GRAPH_CONTROLS
    ], on=["organization_id", "dof"], how="left")
    .merge(robustness, on=["organization_id", "dof"], how="left")
)

display(data.head())



## 2. Organization independence test

A candidate is not useful merely because it correlates with outcomes.  
It should add information beyond ordinary graph invariants.

For each organization candidate, the notebook compares:

- **baseline model:** ordinary graph controls only;
- **extended model:** graph controls plus the organization candidate.

Evaluation uses cross-validation. The reported gain is the change in out-of-fold performance.


In [ ]:

#@title 6. Cross-validated independence tests

O_CANDIDATES = ["O_L1", "O_L2", "O_MAX", "O_PCA1"]

def continuous_cv_r2(frame, target, candidate=None):
    cols = GRAPH_CONTROLS + ([candidate] if candidate else [])
    work = frame[cols + [target]].dropna()
    X = work[cols]
    y = work[target].to_numpy()
    cv = KFold(n_splits=CV_SPLITS, shuffle=True, random_state=SEED)
    model = make_pipeline(
        SimpleImputer(strategy="median"),
        StandardScaler(),
        LinearRegression()
    )
    pred = cross_val_predict(model, X, y, cv=cv)
    return r2_score(y, pred)

def binary_cv_metrics(frame, target, candidate=None):
    cols = GRAPH_CONTROLS + ([candidate] if candidate else [])
    work = frame[cols + [target]].dropna()
    X = work[cols]
    y = work[target].astype(int).to_numpy()

    if len(np.unique(y)) < 2:
        return {"balanced_accuracy": np.nan, "roc_auc": np.nan}

    cv = StratifiedKFold(n_splits=CV_SPLITS, shuffle=True, random_state=SEED)
    model = make_pipeline(
        SimpleImputer(strategy="median"),
        StandardScaler(),
        LogisticRegression(max_iter=3000, class_weight="balanced", random_state=SEED)
    )
    prob = cross_val_predict(model, X, y, cv=cv, method="predict_proba")[:, 1]
    pred = (prob >= 0.5).astype(int)
    return {
        "balanced_accuracy": balanced_accuracy_score(y, pred),
        "roc_auc": roc_auc_score(y, prob),
    }

independence_rows = []

# Continuous outcome: perturbation retention.
base_r2 = continuous_cv_r2(data, "perturbation_retention")
for candidate in O_CANDIDATES:
    ext_r2 = continuous_cv_r2(data, "perturbation_retention", candidate)
    independence_rows.append({
        "candidate": candidate,
        "outcome": "perturbation_retention",
        "metric": "cv_r2",
        "baseline": base_r2,
        "extended": ext_r2,
        "delta": ext_r2 - base_r2,
    })

# Binary outcomes.
for outcome in ["novel_to_lower_dof", "closed", "stable", "primitive_preserved"]:
    base = binary_cv_metrics(data, outcome)
    for candidate in O_CANDIDATES:
        ext = binary_cv_metrics(data, outcome, candidate)
        for metric in ["balanced_accuracy", "roc_auc"]:
            independence_rows.append({
                "candidate": candidate,
                "outcome": outcome,
                "metric": metric,
                "baseline": base[metric],
                "extended": ext[metric],
                "delta": ext[metric] - base[metric],
            })

independence = pd.DataFrame(independence_rows)
display(independence.sort_values("delta", ascending=False).head(25))


In [ ]:

#@title 7. Residual independence from graph controls

residual_rows = []
X_controls = data[GRAPH_CONTROLS]

for candidate in O_CANDIDATES:
    work = data[GRAPH_CONTROLS + [candidate, "dof",
                                  "perturbation_retention"]].dropna()
    model = make_pipeline(
        SimpleImputer(strategy="median"),
        StandardScaler(),
        LinearRegression()
    )
    model.fit(work[GRAPH_CONTROLS], work[candidate])
    predicted = model.predict(work[GRAPH_CONTROLS])
    residual = work[candidate].to_numpy() - predicted

    rho_dof = spearmanr(residual, work["dof"]).statistic
    rho_robust = spearmanr(residual, work["perturbation_retention"]).statistic

    residual_rows.append({
        "candidate": candidate,
        "graph_control_r2": r2_score(work[candidate], predicted),
        "residual_spearman_dof": rho_dof,
        "residual_spearman_robustness": rho_robust,
        "residual_sd": float(np.std(residual, ddof=1)),
    })

residual_report = pd.DataFrame(residual_rows)
display(residual_report)



## 3. Resolution relative to symmetry conditions

Organization is held fixed. A reference condition determines which relational deviations become observable as distinction density.

For a reference weight vector \(w_{\langle S\rangle}\):

\[
\delta D_{\langle S\rangle}(A)
=
\frac{\sum_i w_i |z_i(A)|}
{\sum_i w_i}
\]

The notebook uses four declared reference conditions:

- **uniform:** all structural deviations contribute equally;
- **depth:** depth and branching deviations are emphasized;
- **boundary:** articulation and bridge deviations are emphasized;
- **closure:** SCC, cycle, and transitivity deviations are emphasized.

These are operational probes, not claims that these are the only possible symmetry conditions.


In [ ]:

#@title 8. Define reference conditions and resolve distinction density

REFERENCE_FAMILIES = {
    "S_uniform": list(z_names),
    "S_depth": [
        n for n in z_names
        if any(k in n for k in ["depth", "branching", "leaf"])
    ],
    "S_boundary": [
        n for n in z_names
        if any(k in n for k in ["articulation", "bridge"])
    ],
    "S_closure": [
        n for n in z_names
        if any(k in n for k in ["cycle", "scc", "transitivity"])
    ],
}

for name, cols in REFERENCE_FAMILIES.items():
    if not cols:
        raise ValueError(f"Reference {name} selected no dimensions.")

resolution = organization[
    ["organization_id", "dof", "signature", "O_L1", "O_L2", "O_MAX", "O_PCA1"]
].copy()

for ref_name, cols in REFERENCE_FAMILIES.items():
    resolution[f"deltaD__{ref_name}"] = Z[cols].abs().mean(axis=1)

delta_cols = [c for c in resolution.columns if c.startswith("deltaD__")]
resolution["deltaD_reference_mean"] = resolution[delta_cols].mean(axis=1)
resolution["deltaD_reference_sd"] = resolution[delta_cols].std(axis=1)
resolution["deltaD_reference_range"] = (
    resolution[delta_cols].max(axis=1) - resolution[delta_cols].min(axis=1)
)

display(pd.DataFrame({
    "reference": list(REFERENCE_FAMILIES),
    "dimensions": [len(v) for v in REFERENCE_FAMILIES.values()],
    "members": [", ".join(v) for v in REFERENCE_FAMILIES.values()],
}))
display(resolution.head())


In [ ]:

#@title 9. Test whether reference changes measurement while O remains fixed

reference_variation = {
    "organizations": int(len(resolution)),
    "nonzero_reference_range_count": int(
        (resolution["deltaD_reference_range"] > 1e-12).sum()
    ),
    "nonzero_reference_range_rate": float(
        (resolution["deltaD_reference_range"] > 1e-12).mean()
    ),
    "median_reference_range": float(
        resolution["deltaD_reference_range"].median()
    ),
    "mean_reference_sd": float(
        resolution["deltaD_reference_sd"].mean()
    ),
}

print(json.dumps(reference_variation, indent=2))

sample = resolution.sort_values(
    "deltaD_reference_range", ascending=False
).head(15)
display(sample[
    ["organization_id", "dof", "signature", "O_L1"] +
    delta_cols + ["deltaD_reference_range"]
])


In [ ]:

#@title 10. Visualize fixed organization under multiple references

plot_sample = resolution.sort_values(
    "deltaD_reference_range", ascending=False
).head(12).copy()

long_plot = plot_sample.melt(
    id_vars=["organization_id", "O_L1"],
    value_vars=delta_cols,
    var_name="reference",
    value_name="distinction_density",
)

fig, ax = plt.subplots(figsize=(12, 6))
for org_id, group in long_plot.groupby("organization_id"):
    ax.plot(
        group["reference"].str.replace("deltaD__", "", regex=False),
        group["distinction_density"],
        marker="o",
        alpha=0.65,
        label=org_id,
    )
ax.set_ylabel("Resolved distinction density")
ax.set_xlabel("Reference condition")
ax.set_title("Same organization candidate, different reference resolution")
ax.tick_params(axis="x", rotation=25)
ax.legend(fontsize=7, ncol=2)
ax.grid(True, alpha=0.3)
plt.show()



## 4. Projection axiom test

Candidate axiom:

\[
\mathcal O(P(A)) \leq \mathcal O(A)
\]

The supplied projection table identifies source and projected signatures.  
The notebook joins each signature to its measured organization candidate and tests monotonicity.

A violation is preserved as a counterexample. It is not discarded.


In [ ]:

#@title 11. Join organization candidates onto projection relations

# Duplicate signatures within a DoF are summarized conservatively by their mean.
signature_O = organization.groupby(["dof", "signature"])[O_CANDIDATES].mean().reset_index()

proj = projection.merge(
    signature_O.rename(columns={
        "dof": "source_dof",
        "signature": "source_signature",
        **{c: f"{c}__source" for c in O_CANDIDATES}
    }),
    on=["source_dof", "source_signature"],
    how="left",
)

proj = proj.merge(
    signature_O.rename(columns={
        "dof": "target_dof",
        "signature": "projected_signature",
        **{c: f"{c}__target" for c in O_CANDIDATES}
    }),
    on=["target_dof", "projected_signature"],
    how="left",
)

projection_test_rows = []
projection_counterexamples = []

for candidate in O_CANDIDATES:
    src = f"{candidate}__source"
    tgt = f"{candidate}__target"
    valid = proj[[src, tgt]].notna().all(axis=1)
    work = proj.loc[valid].copy()
    work[f"{candidate}__delta"] = work[tgt] - work[src]
    work[f"{candidate}__monotonic"] = work[tgt] <= work[src] + 1e-12

    projection_test_rows.append({
        "candidate": candidate,
        "matched_projection_rows": int(len(work)),
        "monotonic_count": int(work[f"{candidate}__monotonic"].sum()),
        "violation_count": int((~work[f"{candidate}__monotonic"]).sum()),
        "monotonic_rate": float(work[f"{candidate}__monotonic"].mean())
            if len(work) else np.nan,
        "mean_target_minus_source": float(work[f"{candidate}__delta"].mean())
            if len(work) else np.nan,
        "median_target_minus_source": float(work[f"{candidate}__delta"].median())
            if len(work) else np.nan,
    })

    violated = work.loc[~work[f"{candidate}__monotonic"]].copy()
    violated["candidate"] = candidate
    violated["organization_delta"] = violated[f"{candidate}__delta"]
    projection_counterexamples.append(violated)

projection_axiom_report = pd.DataFrame(projection_test_rows)
projection_counterexamples = (
    pd.concat(projection_counterexamples, ignore_index=True)
    if projection_counterexamples else pd.DataFrame()
)

display(projection_axiom_report)
print("Projection counterexamples:", len(projection_counterexamples))



## 5. Perturbation response: \(\Delta\mathcal O\) versus \(\Delta\delta D\)

Notebook 16 saved perturbation retention but not every perturbed feature vector.  
Therefore, Notebook 17 can test whether organization predicts perturbation retention, but it cannot reconstruct exact \(\Delta\mathcal O\) and \(\Delta\delta D\) for each altered graph.

This limitation is preserved explicitly. A future notebook should save the full perturbed feature vector for every trial.


In [ ]:

#@title 12. Organization–robustness relationships

relationship_rows = []
for candidate in O_CANDIDATES:
    work = data[[candidate, "perturbation_retention", "dof"]].dropna()
    relationship_rows.append({
        "candidate": candidate,
        "spearman_with_retention":
            float(spearmanr(work[candidate], work["perturbation_retention"]).statistic),
        "spearman_with_dof":
            float(spearmanr(work[candidate], work["dof"]).statistic),
    })

organization_relationships = pd.DataFrame(relationship_rows)
display(organization_relationships)



## 6. Axiom audit

The notebook reports each axiom separately:

- **Identity:** computational sanity check.
- **Projection monotonicity:** empirical test with counterexamples.
- **Primitive preservation:** association test, not causal proof.
- **Symmetry/reference resolution:** whether \(\delta D\) varies by reference while \(\mathcal O\) is held fixed.
- **Composition:** not testable from the current archive because explicit composition pairs were not preserved.


In [ ]:

#@title 13. Axiom tests

axiom_results = []

# Identity.
identity_pass = bool(np.allclose(
    organization["O_L1"].to_numpy(),
    organization["O_L1"].copy().to_numpy(),
    equal_nan=True,
))
axiom_results.append({
    "axiom": "identity",
    "status": "PASS_SANITY_CHECK" if identity_pass else "FAIL",
    "evidence": "O(A) equals copied O(A).",
    "claim_type": "computational_sanity",
})

# Projection monotonicity, candidate by candidate.
for row in projection_axiom_report.itertuples(index=False):
    axiom_results.append({
        "axiom": f"projection_monotonicity__{row.candidate}",
        "status": (
            "SUPPORTED_WITHIN_MATCHED_ROWS"
            if row.violation_count == 0
            else "COUNTEREXAMPLES_FOUND"
        ),
        "evidence": {
            "matched_rows": int(row.matched_projection_rows),
            "monotonic_rate": row.monotonic_rate,
            "violations": int(row.violation_count),
        },
        "claim_type": "bounded_empirical",
    })

# Primitive preservation association.
for candidate in O_CANDIDATES:
    preserved = data.loc[data["primitive_preserved"] == True, candidate].dropna()
    not_preserved = data.loc[data["primitive_preserved"] == False, candidate].dropna()
    if len(preserved) and len(not_preserved):
        test = mannwhitneyu(preserved, not_preserved, alternative="two-sided")
        evidence = {
            "preserved_mean": float(preserved.mean()),
            "not_preserved_mean": float(not_preserved.mean()),
            "mannwhitney_p": float(test.pvalue),
        }
        status = "ASSOCIATION_TESTED"
    else:
        evidence = {"reason": "Both primitive classes were not represented."}
        status = "NOT_TESTABLE"
    axiom_results.append({
        "axiom": f"primitive_preservation__{candidate}",
        "status": status,
        "evidence": evidence,
        "claim_type": "association_not_causation",
    })

# Reference resolution.
axiom_results.append({
    "axiom": "organization_resolves_relative_to_reference",
    "status": (
        "SUPPORTED_OPERATIONALLY"
        if reference_variation["nonzero_reference_range_rate"] > 0
        else "NOT_OBSERVED"
    ),
    "evidence": reference_variation,
    "claim_type": "operational_definition_test",
})

# Composition unavailable.
axiom_results.append({
    "axiom": "nonadditive_composition",
    "status": "NOT_TESTABLE_FROM_CURRENT_ARCHIVE",
    "evidence": (
        "Notebook 16 did not preserve explicit A, B, and composed A⊗B identities."
    ),
    "claim_type": "missing_required_data",
})

display(pd.DataFrame(axiom_results))


In [ ]:

#@title 14. Rank organization candidates

# Rank candidates by several bounded criteria.
rank_rows = []

for candidate in O_CANDIDATES:
    ind = independence[independence["candidate"] == candidate]
    residual = residual_report[residual_report["candidate"] == candidate].iloc[0]
    proj_row = projection_axiom_report[
        projection_axiom_report["candidate"] == candidate
    ].iloc[0]
    rel = organization_relationships[
        organization_relationships["candidate"] == candidate
    ].iloc[0]

    positive_deltas = ind["delta"].replace([np.inf, -np.inf], np.nan).dropna()
    rank_rows.append({
        "candidate": candidate,
        "mean_incremental_predictive_delta": float(positive_deltas.mean()),
        "max_incremental_predictive_delta": float(positive_deltas.max()),
        "graph_control_r2": float(residual["graph_control_r2"]),
        "unexplained_fraction": float(1 - residual["graph_control_r2"]),
        "projection_monotonic_rate": float(proj_row["monotonic_rate"]),
        "spearman_robustness": float(rel["spearman_with_retention"]),
        "spearman_dof": float(rel["spearman_with_dof"]),
    })

rankings = pd.DataFrame(rank_rows)

# Transparent heuristic score; raw components remain available.
rankings["discovery_score"] = (
    rankings["mean_incremental_predictive_delta"].fillna(0)
    + 0.25 * rankings["unexplained_fraction"].clip(lower=0)
    + 0.25 * rankings["projection_monotonic_rate"].fillna(0)
    + 0.25 * rankings["spearman_robustness"].abs().fillna(0)
)

rankings = rankings.sort_values("discovery_score", ascending=False)
display(rankings)


In [ ]:

#@title 15. Preserve counterexamples

# Higher DoF but lower candidate organization than at least one lower-DoF case.
dof_counterexample_rows = []

for candidate in O_CANDIDATES:
    running_max = -np.inf
    lower_max = {}
    for dof in sorted(organization["dof"].unique()):
        lower_max[dof] = running_max
        vals = organization.loc[organization["dof"] == dof, candidate].dropna()
        if len(vals):
            running_max = max(running_max, vals.max())

    temp = organization.copy()
    temp["candidate"] = candidate
    temp["candidate_value"] = temp[candidate]
    temp["lower_dof_max"] = temp["dof"].map(lower_max)
    temp["counterexample"] = temp["candidate_value"] < temp["lower_dof_max"]
    dof_counterexample_rows.append(temp.loc[temp["counterexample"], [
        "organization_id", "dof", "signature", "candidate",
        "candidate_value", "lower_dof_max"
    ]])

dof_counterexamples = pd.concat(dof_counterexample_rows, ignore_index=True)

print("DoF monotonicity counterexamples:", len(dof_counterexamples))
print("Projection monotonicity counterexamples:", len(projection_counterexamples))
display(dof_counterexamples.head(20))


In [ ]:

#@title 16. Bounded interpretation protocol

best = rankings.iloc[0]
best_candidate = best["candidate"]

best_projection = projection_axiom_report.loc[
    projection_axiom_report["candidate"] == best_candidate
].iloc[0]

best_independence = independence[
    independence["candidate"] == best_candidate
]["delta"].dropna()

if (
    best["unexplained_fraction"] > 0.10
    and best_independence.max() > 0
):
    organization_status = "CANDIDATE_RETAINS_NONREDUNDANT_SIGNAL"
else:
    organization_status = "CANDIDATE_NOT_YET_DISTINCT_FROM_GRAPH_INVARIANTS"

if reference_variation["nonzero_reference_range_rate"] > 0.95:
    resolution_status = "REFERENCE_DEPENDENT_RESOLUTION_OBSERVED"
else:
    resolution_status = "REFERENCE_DEPENDENCE_PARTIAL_OR_ABSENT"

if best_projection["monotonic_rate"] >= 0.95:
    projection_status = "PROJECTION_AXIOM_STRONGLY_SUPPORTED_BOUNDED"
elif best_projection["monotonic_rate"] >= 0.75:
    projection_status = "PROJECTION_AXIOM_PARTIALLY_SUPPORTED"
else:
    projection_status = "PROJECTION_AXIOM_NOT_SUPPORTED"

decision_record = {
    "campaign_id": CAMPAIGN_ID,
    "best_operational_candidate": best_candidate,
    "organization_status": organization_status,
    "resolution_status": resolution_status,
    "projection_status": projection_status,
    "interpretation": (
        "Organization candidates and reference-resolved distinction densities "
        "are empirically separable within the supplied Notebook 16 domain. "
        "This remains an operational construction, not a formal primitive."
    ),
    "claim_ceiling": "C2_BOUNDED_NOTEBOOK_OUTPUT_AFTER_GOVERNED_REVIEW",
    "promotion_block": True,
    "non_claims": [
        "No proof that organization is ontologically primitive.",
        "No proof that the selected candidate is unique.",
        "No implementation-independent organization calculus.",
        "No external physical validation.",
        "No universal projection monotonicity claim.",
    ],
}

print(json.dumps(decision_record, indent=2))


In [ ]:

#@title 17. Export the complete result archive

def write_json(path, value):
    Path(path).write_text(json.dumps(value, indent=2, default=str))

def sha256(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest().upper()

organization.to_csv(OUTPUT_DIR / "organization_candidates.csv", index=False)
data.to_csv(OUTPUT_DIR / "organization_joined_data.csv", index=False)
explained.to_csv(OUTPUT_DIR / "organization_pca_variance.csv", index=False)
independence.to_csv(OUTPUT_DIR / "organization_independence_results.csv", index=False)
residual_report.to_csv(OUTPUT_DIR / "organization_residual_independence.csv", index=False)
resolution.to_csv(OUTPUT_DIR / "reference_resolution_table.csv", index=False)
projection_axiom_report.to_csv(
    OUTPUT_DIR / "projection_axiom_report.csv", index=False
)
projection_counterexamples.to_csv(
    OUTPUT_DIR / "projection_counterexamples.csv", index=False
)
organization_relationships.to_csv(
    OUTPUT_DIR / "organization_robustness_relationships.csv", index=False
)
rankings.to_csv(OUTPUT_DIR / "organization_metric_rankings.csv", index=False)
dof_counterexamples.to_csv(
    OUTPUT_DIR / "organization_dof_counterexamples.csv", index=False
)

write_json(OUTPUT_DIR / "reference_conditions.json", REFERENCE_FAMILIES)
write_json(OUTPUT_DIR / "reference_variation_report.json", reference_variation)
write_json(OUTPUT_DIR / "organization_axiom_tests.json", axiom_results)
write_json(OUTPUT_DIR / "decision_record.json", decision_record)

output_files = [
    path for path in OUTPUT_DIR.iterdir()
    if path.is_file() and path.name != "manifest.json"
]

manifest = {
    "campaign_id": CAMPAIGN_ID,
    "status": "EXECUTED",
    "created_at": datetime.now(timezone.utc).isoformat(),
    "seed": SEED,
    "source_archive": source_zip.name,
    "source_campaign": source_manifest.get("campaign_id"),
    "organizations_loaded": int(len(features)),
    "projection_rows_loaded": int(len(projection)),
    "perturbation_rows_loaded": int(len(perturbations)),
    "organization_candidates": O_CANDIDATES,
    "reference_conditions": REFERENCE_FAMILIES,
    "operational_statement": (
        "Organization candidate is computed independently of the selected "
        "reference; distinction density is then resolved from matched-null "
        "structural deviations under declared reference weights."
    ),
    "known_limitation": (
        "Exact perturbation-level delta-O and delta-distinction-density cannot "
        "be computed because Notebook 16 did not preserve perturbed feature vectors."
    ),
    "decision": decision_record,
    "claim_ceiling": "C2_BOUNDED_NOTEBOOK_OUTPUT_AFTER_GOVERNED_REVIEW",
    "output_files": {},
}

for path in output_files:
    manifest["output_files"][path.name] = {
        "sha256": sha256(path),
        "bytes": path.stat().st_size,
    }

write_json(OUTPUT_DIR / "manifest.json", manifest)

bundle = Path("/content") / f"{CAMPAIGN_ID}_RESULTS.zip"
if bundle.exists():
    bundle.unlink()

with zipfile.ZipFile(bundle, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for path in sorted(OUTPUT_DIR.iterdir()):
        if path.is_file():
            zf.write(path, arcname=path.name)

print("Results:", bundle)
print("SHA-256:", sha256(bundle))
print("Files:", len(list(OUTPUT_DIR.iterdir())))


In [ ]:

#@title 18. Download results in Colab
try:
    from google.colab import files
    files.download(str(bundle))
except ImportError:
    print("Not running in Colab. Results are at:", bundle)



# Interpretation guide

## Result A — organization candidate adds predictive information

Interpretation:

> Within the bounded domain, the candidate is not fully reducible to the selected ordinary graph controls.

This is not proof that organization is a new primitive.

## Result B — reference conditions change \(\delta D\) while \(\mathcal O\) remains fixed

Interpretation:

> The operational distinction between organization and resolved distinction density is computationally coherent.

This follows partly from the declared measurement construction and must later produce independent predictions.

## Result C — projection monotonicity is high but imperfect

Interpretation:

> Projection usually reduces or preserves the candidate organization measure, but preserved counterexamples block a universal axiom.

Investigate whether the violations reflect:

- a poor organization metric;
- a poor projection operator;
- organization-generating coarse graining;
- signature collisions.

## Result D — graph controls explain nearly all organization candidates

Interpretation:

> The current candidates remain repackaged graph invariants. H₀ is not rejected.

## Required Notebook 18 improvement

Preserve the complete feature vector for every perturbed graph so that the next notebook can measure:

\[
\Delta\mathcal O
\quad\text{and}\quad
\Delta\delta D_{\langle S\rangle}
\]

directly for each perturbation.
